# Telemetry AD Results Notebook
This notebook visualizes baseline results for NAB and SKAB using generated report files.

Expected input files:
- `reports/experiments/nab_series_summary.csv`
- `reports/experiments/threshold_sweep_summary.csv`
- `reports/<dataset>/<variant>/metrics.json`
- `reports/<dataset>/<variant>/predictions.csv`


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
REPORTS = ROOT / 'reports'
EXP = REPORTS / 'experiments'
print('ROOT:', ROOT)


## 1) Summary Tables

In [ ]:
nab_summary = pd.read_csv(EXP / 'nab_series_summary.csv')
threshold_sweep = pd.read_csv(EXP / 'threshold_sweep_summary.csv')
display(nab_summary.sort_values(['series', 'model']))
display(threshold_sweep.sort_values(['dataset', 'model', 'threshold_percentile']))


## 2) NAB Per-Series Model Comparison (F1)

In [ ]:
pivot_f1 = nab_summary.pivot(index='series', columns='model', values='f1').fillna(0.0)
ax = pivot_f1.plot(kind='bar', figsize=(10, 4), rot=20)
ax.set_ylabel('F1 score')
ax.set_title('NAB per-series F1 by model')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 3) Threshold Sweep Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for i, ds in enumerate(['nab', 'skab']):
    sub = threshold_sweep[threshold_sweep['dataset'] == ds]
    for model in sorted(sub['model'].unique()):
        msub = sub[sub['model'] == model].sort_values('threshold_percentile')
        axes[i].plot(msub['threshold_percentile'], msub['f1'], marker='o', label=model)
    axes[i].set_title(f'{ds.upper()} threshold sweep')
    axes[i].set_xlabel('threshold_percentile')
    axes[i].set_ylabel('F1')
    axes[i].grid(alpha=0.3)
    axes[i].legend()
plt.tight_layout()
plt.show()


## 4) Helper Functions for Detailed Inspection

In [ ]:
def load_metrics(dataset: str, variant: str) -> dict:
    with (REPORTS / dataset / variant / 'metrics.json').open('r', encoding='utf-8') as f:
        return json.load(f)

def load_predictions(dataset: str, variant: str) -> pd.DataFrame:
    return pd.read_csv(REPORTS / dataset / variant / 'predictions.csv')

def plot_timeseries_predictions(dataset: str, variant: str, max_points: int = 1200):
    pred = load_predictions(dataset, variant).copy()
    if len(pred) > max_points:
        pred = pred.iloc[-max_points:]

    if 'timestamp' in pred.columns:
        try:
            pred['timestamp'] = pd.to_datetime(pred['timestamp'])
        except Exception:
            pass

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    for model in ['zscore', 'iforest']:
        score_col = f'{model}_score'
        pred_col = f'{model}_pred'
        if score_col in pred.columns:
            axes[0].plot(pred.index, pred[score_col], label=score_col, linewidth=1.0)
        if pred_col in pred.columns:
            idx = pred.index[pred[pred_col] == 1]
            if len(idx):
                axes[1].scatter(idx, np.ones(len(idx)) if model == 'zscore' else np.full(len(idx), 0.8), s=10, label=pred_col)

    if 'y_true' in pred.columns:
        idx_true = pred.index[pred['y_true'] == 1]
        if len(idx_true):
            axes[1].scatter(idx_true, np.full(len(idx_true), 0.2), s=10, label='y_true')

    axes[0].set_title(f'{dataset}:{variant} anomaly scores')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].set_title('Predicted vs true anomaly markers')
    axes[1].set_yticks([])
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

def plot_confusion(dataset: str, variant: str):
    m = load_metrics(dataset, variant)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    for i, model in enumerate(['zscore', 'iforest']):
        cm = np.array(m[model]['confusion_matrix'])
        im = axes[i].imshow(cm, cmap='Blues')
        axes[i].set_title(f'{model} CM')
        axes[i].set_xlabel('Pred')
        axes[i].set_ylabel('True')
        for r in range(cm.shape[0]):
            for c in range(cm.shape[1]):
                axes[i].text(c, r, int(cm[r, c]), ha='center', va='center', color='black')
    plt.tight_layout()
    plt.show()


## 5) Detailed Examples

In [ ]:
plot_confusion('nab', 'ec2_cpu_utilization_5f5533')
plot_timeseries_predictions('nab', 'ec2_cpu_utilization_5f5533')


In [ ]:
plot_confusion('skab', 'anomalyfree_vs_valve1_1')
plot_timeseries_predictions('skab', 'anomalyfree_vs_valve1_1')


## 6) Notes
- Z-score was more sensitive on some NAB series but failed on SKAB with current feature/threshold setup.
- Isolation Forest benefited from threshold tuning (NAB best near 99.9, SKAB best near 99.0).
- `ec2_network_in_257a54` remains challenging and should be targeted for feature/parameter tuning.
